In [1]:
import os
# os.environ["PYTORCH_ENABLE_MPS_FALLBACK"]="1" # Keep if needed for MPS # does not work for some reason
import torch
from deeplsd.models.deeplsd_inference import DeepLSD
from line_understanding.json_saver import save_lines_to_json 


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
conf = {'detect_lines': True, 'line_detection_params': {'merge': False, 'filtering': True, 'grad_thresh': 3}}
ckpt = torch.load('../weights/deeplsd_md.tar', map_location='cpu', weights_only=False)
net = DeepLSD(conf)
net.load_state_dict(ckpt['model'])
net = net.to(device).eval()

In [10]:
from line_understanding.pipeline import process_image_pipeline, plot_pipeline_results
from upload_hypersim import upload_images, delete_images, generate_image_list
frame_str = "0000"
start_image = "ai_001_001"
num_images = 100 # number of images per scene
num_scenes = 2
num_samples = num_images * num_scenes
desired_images = generate_image_list(start_image, num_scenes)
print("Will download and process {} images".format(num_samples))
files_to_download = [
    f"frame.{frame_str}.color.jpg",
    f"frame.{frame_str}.depth_meters.hdf5",
    f"frame.{frame_str}.normal_world.hdf5",
    f"frame.{frame_str}.position.hdf5"
]
print("Starting to Download Images.")
# upload_images(desired_images, files_to_download)
print("Finished Dowloading Images. Starting the ground truth generation.")
image_data = {} # dictionary of dictionaries: key is image_id and value is processed_data dictionary
print(desired_images)
for (i,image_id) in enumerate(desired_images):
    # processed_data = process_image_pipeline(image_id, frame_str, net, device, depth_thresh=50, normal_thresh=1.5 * 1e7, cluster_selection_epsilon=0.002, min_cluster_size=25,plot_imgs = False)
    print("Processing scene {} out of {}".format(i+1,num_scenes))
    for num_image in range(0,num_images):
        print("Processing image {} out of {} of scene {}/{}".format(num_image+1,num_images,i+1,num_scenes))
        if num_image < 10:
            frame_str = "000{}".format(num_image)
        else:
            frame_str = "00{}".format(num_image)
        processed_data = process_image_pipeline(image_id, frame_str, net, device, depth_thresh=3500, normal_thresh=(1.25 * 1e7)**2, cluster_selection_epsilon=0.01,min_cluster_size=10,plot_imgs = False)
        image_data[image_id + '.' + frame_str] = processed_data
        


# process image pipeline
"""
Each key in image_data contains
    {
        "image_dir": image_dir,
        "composite_after": composite_after,
        "pred_lines": pred_lines,  
        "img": img,
        "normals": normals,
        "world_coordinates": world_coordinates,
        "plane_map": plane_map,
        "segmentation_map": segmentation_map,
        "original_map": original_map,
        "coplanarity_labels": all_coplanarity_labels,  # if needed
        "line_info": line_info,
        "coplanarity_matrix": coplanarity_matrix,
        "scores": scores,
        "original_lines": original_lines    
    }
"""
    

Will download and process 20 images
Starting to Download Images.
Finished Dowloading Images. Starting the ground truth generation.
['ai_001_001', 'ai_001_002', 'ai_001_003', 'ai_001_004', 'ai_001_005', 'ai_001_006', 'ai_001_007', 'ai_001_008', 'ai_001_009', 'ai_002_000', 'ai_002_001', 'ai_002_002', 'ai_002_003', 'ai_002_004', 'ai_002_005', 'ai_002_006', 'ai_002_007', 'ai_002_008', 'ai_002_009', 'ai_003_000']
Processing scene 1 out of 20
Processing image 1 out of 1 of scene 1/20
data/ai_001_001/ai_001_001/images/scene_cam_00_final_preview/frame.0000.color.jpg
data/ai_001_001/ai_001_001/images/scene_cam_00_geometry_hdf5/frame.0000.normal_world.hdf5
data/ai_001_001/ai_001_001/images/scene_cam_00_geometry_hdf5/frame.0000.depth_meters.hdf5
data/ai_001_001/ai_001_001/images/scene_cam_00_geometry_hdf5/frame.0000.position.hdf5
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
F

KeyboardInterrupt: 

In [7]:
import numpy as np
import math
import copy

print("Creating a backup copy 'image_data_old'...")
image_data_old = copy.deepcopy(image_data)
print("Backup created.")
keys_to_delete = [] # List to store keys for deletion


print("Starting NaN check in scores...")

nan_found_overall = False # Keep track if any NaN was found across all items

# Use list(image_data.items()) to iterate over a static copy of items
for key, item in image_data.items():
    nan_found_in_this_item = False # Flag for the current item

    # Check if 'scores' key exists and is not None
    if 'scores' in item and item['scores'] is not None:
        local_scores = item['scores']
        nan_indices = [] # Reset for each item

        # Check if local_scores is iterable
        if hasattr(local_scores, '__iter__'):
            try:
                # Find indices where the score is NaN
                for index, score in enumerate(local_scores):
                    if isinstance(score, (int, float, np.number)) and np.isnan(score):
                         nan_indices.append(index)
                         nan_found_in_this_item = True # Added: Mark item for deletion
                         # break # Uncomment if you don't need the full list of nan_indices for printing

            except TypeError as e:
                 print(f"Warning for Image ID {key}: Error processing scores. Maybe non-numeric data? Error: {e}")
                 continue # Skip to next image

            # If the nan_indices list is not empty, NaNs were found
            if nan_indices: # This condition is met if nan_found_in_this_item is True
                print(f"--- NaN scores found for Image ID: {key} ---")
                print(f"  Indices with NaN: {nan_indices}")
                nan_found_overall = True
                if key not in keys_to_delete: # Avoid duplicates if break wasn't used
                    keys_to_delete.append(key)
                    print(f"    Marked for deletion: {key}")

        else:
            # Handle cases where 'scores' is not iterable (e.g., a single number)
             print(f"Info for Image ID {key}: 'scores' field is not iterable (e.g., not a list). Value: {local_scores}")
             try:
                 if isinstance(local_scores, (int, float, np.number)) and np.isnan(local_scores):
                     print(f"--- NaN score found for Image ID: {key} ---")
                     print(f"  Score value is NaN.")
                     nan_found_overall = True
                     nan_found_in_this_item = True # Added: Mark item for deletion
                     if key not in keys_to_delete:
                         keys_to_delete.append(key)
                         print(f"    Marked for deletion: {key}")
             except TypeError:
                  print(f"Warning for Image ID {key}: Could not check non-iterable score for NaN. Value: {local_scores}")


    elif 'scores' not in item:
        print(f"Warning for Image ID {key}: 'scores' key is missing.")
    else: # Handles item['scores'] is None
        print(f"Info for Image ID {key}: 'scores' is None.")


if not nan_found_overall:
    print("\nNo NaN entries found in any 'scores' lists. No deletions performed.")
else:
    print("\nNaN check complete.")
    print(f"\nDeleting {len(keys_to_delete)} entries with NaN scores from 'image_data'...")
    for key_to_del in keys_to_delete:
        if key_to_del in image_data: # Check if it wasn't already deleted somehow
            del image_data[key_to_del]
    print("Deletion complete.")
    print(f"Original data remains in 'image_data_old' (size: {len(image_data_old)})")
    print(f"Current 'image_data' size: {len(image_data)}")

Creating a backup copy 'image_data_old'...
Backup created.
Starting NaN check in scores...

No NaN entries found in any 'scores' lists. No deletions performed.


In [6]:
import numpy as np
import cv2
import torch
from torchvision.ops import roi_align
import matplotlib.pyplot as plt
# from tqdm import tqdm for normal python script not notebook!
from tqdm.notebook import tqdm # only for notebook!

def plot_lines(lines, img1, img2):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Plot the first image with lines
    axes[0].imshow(img1.permute(1, 2, 0))
    axes[1].imshow(img2.permute(1, 2, 0))
    for line in lines:
        axes[0].plot([line[0, 0], line[1, 0]], [line[0, 1], line[1, 1]], color="red", linewidth=2)
        axes[1].plot([line[0, 0], line[1, 0]], [line[0, 1], line[1, 1]], color="red", linewidth=2)
    axes[0].axis("off")
    axes[1].axis("off")
    
    plt.show()

def to_numpy(img):
    """
    Convert an image to a NumPy array with shape (H, W, C).
    If the image is a torch.Tensor with shape [1, H, W] (grayscale), convert it
    to a NumPy array of shape (H, W, 1). If it's already a numpy array, just return a copy.
    """
    if isinstance(img, torch.Tensor):
        arr = img.cpu().numpy()  # e.g. shape: (1, H, W) or (C, H, W)
        if arr.ndim == 3 and arr.shape[0] == 1:
            # Grayscale tensor: squeeze the channel dimension then add as last dim.
            arr = np.squeeze(arr, axis=0)  # now shape: (H, W)
            arr = np.expand_dims(arr, axis=-1)  # now shape: (H, W, 1)
        elif arr.ndim == 3 and arr.shape[0] in [3,1]:
            # For multi-channel tensor in (C, H, W) format, transpose to (H, W, C)
            arr = np.transpose(arr, (1, 2, 0))
        return arr.copy()
    else:
        return img.copy()
    


def vectorized_point_line_distance(H, W, x1, y1, x2, y2):
    """
    Compute a distance map of shape (H, W) where each element is the distance 
    from that pixel coordinate (x,y) to the line segment (x1,y1)-(x2,y2).
    """
    # Create a grid of (x,y) coordinates (note: x corresponds to columns, y to rows)
    xs = np.arange(W)
    ys = np.arange(H)
    xv, yv = np.meshgrid(xs, ys)
    
    # Vector from A (x1, y1) to every pixel: shape (H, W)
    APx = xv - x1
    APy = yv - y1
    
    # Vector from A to B
    ABx = x2 - x1
    ABy = y2 - y1
    AB_norm_sq = ABx**2 + ABy**2
    
    # Avoid division by zero for degenerate line segments
    if AB_norm_sq == 0:
        return np.sqrt(APx**2 + APy**2)
    
    # Compute the projection parameter t for every pixel
    t = (APx * ABx + APy * ABy) / AB_norm_sq
    t = np.clip(t, 0, 1)
    
    # Compute the projection point on the line for every pixel
    proj_x = x1 + t * ABx
    proj_y = y1 + t * ABy
    
    # Compute the distance from every pixel to its projection
    dist = np.sqrt((xv - proj_x)**2 + (yv - proj_y)**2)
    return dist

def extract_line_feature_ROIAlign(line, img1, img2, img3, 
                                  plot_results=False, thickness=50, output_size=(32,32)):
    """
    Extracts an ROI from three images based on the bounding box of a line.
    The entire image is masked such that pixels further than `thickness` pixels 
    from the line are set to zero.
    
    Parameters:
        line (np.ndarray): Array with shape (2,2) representing endpoints [[x1,y1], [x2,y2]].
        img1, img2, img3 (np.ndarray or torch.Tensor): Input images.
        plot_results (bool): Whether to display a 3x3 plot (original, masked, ROIAlign for each image).
        thickness (int): Pixel distance threshold; pixels farther away are zeroed.
        output_size (tuple): Desired ROIAlign output size (height, width).
    
    Returns:
        roi_results (torch.Tensor): Tensor of shape (3, C, output_size[0], output_size[1])
                                    resulting from ROIAlign for each image.
    """
    # Convert inputs to numpy arrays if needed (each becomes (H,W,C))
    img1_np = to_numpy(img1)
    img2_np = to_numpy(img2)
    img3_np = to_numpy(img3)
    
    # Use the image dimensions (assuming all images share the same shape)
    H, W = img1_np.shape[:2]
    
    # Compute full-image distance map from each pixel to the line
    # Note: We assume the line endpoints are provided in (x,y) order.
    x1, y1 = line[0]
    x2, y2 = line[1]
    distance_map = vectorized_point_line_distance(H, W, x1, y1, x2, y2)
    
    # Create a full-image mask: True for pixels within the threshold (keep them), False otherwise.
    full_mask = distance_map <= thickness
    
    # For ROI box, we still use the bounding box of the line (with a small margin)
    x_coords = line[:, 0]
    y_coords = line[:, 1]
    margin = 2
    min_x = max(int(np.floor(x_coords.min())) - margin, 0)
    min_y = max(int(np.floor(y_coords.min())) - margin, 0)
    max_x = min(int(np.ceil(x_coords.max())) + margin, W - 1)
    max_y = min(int(np.ceil(y_coords.max())) + margin, H - 1)
    
    # Prepare images and corresponding drawn (line overlay) versions.
    # For "drawn" versions we overlay the line on the original (unmasked) image.
    images = [img1_np.copy(), img2_np.copy(), img3_np.copy()]
    drawn_images = []
    masked_images = []
    
    for im in images:
        # If the image is grayscale, convert to BGR so we can draw a colored line.
        if im.shape[2] == 1:
            im_for_draw = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
        else:
            im_for_draw = im.copy()
        pt1 = tuple(np.round(line[0]).astype(int))
        pt2 = tuple(np.round(line[1]).astype(int))
        cv2.line(im_for_draw, pt1, pt2, color=(255, 0, 0), thickness=2)
        drawn_images.append(im_for_draw)
        
        # Create the masked version by applying the full image mask.
        # If the image is multi-channel, apply the mask on every channel.
        if im.ndim == 3 and im.shape[2] > 1:
            masked = im.copy()
            masked[~full_mask, :] = 0
        else:
            masked = im.copy()
            masked[~full_mask] = 0
        masked_images.append(masked)

    
    # Convert masked images to PyTorch tensors in (C, H, W) order.
    tensor_list = []
    for im in masked_images:
        # If image is in HWC and has only 1 channel, keep it that way
        if im.ndim == 2:
            im = np.expand_dims(im, axis=-1)
        # Convert to float32 tensor and change to (C, H, W)
        tensor_img = torch.from_numpy(im.astype(np.float32)).permute(2, 0, 1)
        tensor_list.append(tensor_img)
    
    # Determine the target number of channels: if any image has 3 channels, use 3.
    target_channels = 3 if any(t.shape[0] == 3 for t in tensor_list) else tensor_list[0].shape[0]
    
    # Adjust channel count: for any tensor with 1 channel but target_channels is 3, repeat the channel.
    for i in range(len(tensor_list)):
        C, H_img, W_img = tensor_list[i].shape
        if C != target_channels:
            if C == 1 and target_channels == 3:
                tensor_list[i] = tensor_list[i].repeat(3, 1, 1)
            else:
                if C > target_channels:
                    tensor_list[i] = tensor_list[i][:target_channels]
                else:
                    pad = torch.zeros((target_channels - C, H_img, W_img), dtype=tensor_list[i].dtype)
                    tensor_list[i] = torch.cat([tensor_list[i], pad], dim=0)
    
    input_tensor = torch.stack(tensor_list, dim=0)  # shape: (3, C, H, W)
    
    # Build ROI boxes based on the computed bounding box of the line (expanded slightly)
    boxes = []
    for i in range(len(images)):
        boxes.append([i, float(min_x), float(min_y), float(max_x + 1), float(max_y + 1)])
    boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
    
    roi_results = roi_align(input_tensor, boxes_tensor, output_size=output_size, spatial_scale=1.0)
        
    # Plot results if requested.
    if plot_results:
        # Create a 3x3 grid of subplots.
        fig, axes = plt.subplots(3, 3, figsize=(16, 16))
        for row in range(3):
            for col in range(3):
                if row == 0:
                    # Original image with line drawn.
                    if col == 0:
                        drawn_tensor = torch.from_numpy(drawn_images[col].astype(np.float32)).permute(2, 0, 1) / 255.0
                        axes[row, col].imshow(drawn_tensor.permute(1, 2, 0).cpu().numpy())
                        axes[row, col].set_title(f'Image {col+1}: Original with Line')
                    elif col == 1:
                        # Plot the distance field image with line
                        axes[row,col].imshow(img2.permute(1, 2, 0))
                        axes[row,col].plot([line[0, 0], line[1, 0]], [line[0, 1], line[1, 1]], color="red", linewidth=2)
                        axes[row, col].set_title(f'Distance Field {col+1}: Original with Line')
                    elif col == 2:
                        # Plot the angle field image with line
                        axes[row,col].imshow(img3.permute(1, 2, 0))
                        axes[row,col].plot([line[0, 0], line[1, 0]], [line[0, 1], line[1, 1]], color="red", linewidth=2)
                        axes[row, col].set_title(f'Angle Field {col+1}: Original with Line')
    
                elif row == 1:
                    # Masked image.
                    masked_tensor = torch.from_numpy(masked_images[col].astype(np.float32)).permute(2, 0, 1) / 255.0
                    axes[row, col].imshow(masked_tensor.permute(1, 2, 0).cpu().numpy())
                    axes[row,col].plot([line[0, 0], line[1, 0]], [line[0, 1], line[1, 1]], color="red", linewidth=2)
                    axes[row, col].set_title(f'Image {col+1}: Masked')
                elif row == 2:
                    # ROIAlign result.
                    roi_disp = roi_results[col].clone()
                    # Normalize for display.
                    roi_disp = roi_disp - roi_disp.min()
                    if roi_disp.max() > 0:
                        roi_disp = roi_disp / roi_disp.max()
                    axes[row, col].imshow(roi_disp.permute(1, 2, 0).cpu().numpy())
                    axes[row, col].set_title(f'Image {col+1}: ROIAlign')
                axes[row, col].axis('off')
        plt.tight_layout()
        plt.show()

    
    return roi_results


out_H = 64
ratio = 64
output_size = (out_H, out_H)
for key, item in tqdm(image_data.items(), desc="Extracting Features from images",position=0,leave=True):
    print()
    img_tensor = torch.from_numpy(image_data[key]['img'].transpose(2, 0, 1).copy()).unsqueeze(0).float()  # shape: (1, 3, H, W)
    img = image_data[key]['img']
    gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    inputs = {'image': torch.tensor(gray_img, dtype=torch.float32, device=device)[None, None] / 255.}
    with torch.no_grad():
        out = net(inputs)
        pred_lines = out['lines'][0]
        distance_field_norm = out['df_norm']
        angle_field = out["line_level"]
       #print(pred_lines.shape)
        #print(angle_field.shape)
        #print(distance_field_norm.size())
        #print(image_data[key]['img'].shape)
        # plot_lines(pred_lines,distance_field_norm,angle_field)
        # print(pred_lines)
    features = {}
    for i, line in enumerate(tqdm(item['line_info'], desc=f"Applying ROI_Align for line", position=1, leave=False)):
        line["base_line"] = np.asarray(line["base_line"])
        roi_align_patches = extract_line_feature_ROIAlign(line["base_line"],image_data[key]['img'],distance_field_norm,angle_field, plot_results=False, thickness=64, output_size=output_size,)
        line["features"] = roi_align_patches
    

Extracting Features from images:   0%|          | 0/8 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
for key,item in image_data.items():
    print(len(item['line_info']))
    print(type(item['line_info']))

In [19]:
import torch
import numpy as np
from torch_geometric.data import Dataset, Data
from torch_geometric.utils import dense_to_sparse
from torch_geometric.loader import DataLoader # Use PyG DataLoader
import pytorch_lightning as pl
from torch.utils.data import random_split, Subset # For splitting
from tqdm import tqdm
import logging
import pickle

# Configure logging (ensure it's configured globally or within your main script)
# logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Assume LineGraphRegressionDatasetInMemory class is defined here ---
# (Copying it from the previous response for completeness)
class LineGraphRegressionDataset(Dataset):
    """
    PyTorch Geometric Dataset for graph regression on line features.
    (Copied from previous response - Keeps data in memory, handles processing/saving/loading)
    """
    def __init__(self, root, image_data, load_precomputed_dataset=False):
        super().__init__(root)
        self.root = root
        self.image_data = image_data
        self._processed_file_path = os.path.join(self.root, 'processed_graphs.pkl')
        self.processed_data = []

        if load_precomputed_dataset:
            if os.path.exists(self._processed_file_path):
                logging.info(f"Dataset: Attempting to load precomputed data from: {self._processed_file_path}")
                self.load_dataset(self._processed_file_path)
            else:
                logging.warning(f"Dataset: load_precomputed_dataset=True but file not found at: {self._processed_file_path}. Processing from image_data instead.")
                self._process_data()
        else:
            # Process only if the file doesn't already exist OR if explicitly told not to load precomputed
            # This prevents reprocessing if the file exists but load_precomputed_dataset was False
            if not os.path.exists(self._processed_file_path):
                 logging.info("Dataset: No precomputed file found. Processing dataset from provided image_data...")
                 self._process_data()
            else:
                 logging.info(f"Dataset: Precomputed file exists at {self._processed_file_path}, but load_precomputed_dataset=False. Reprocessing anyways and overwriting.")
                 # Load the existing data even if load_precomputed_dataset is False,
                 # assuming the user wants to use the existing processed data unless it's missing.
                 # self.load_dataset(self._processed_file_path)
                 # If you strictly want to reprocess whenever load_precomputed_dataset is False,
                 # uncomment the next line and comment out the load_dataset call above.
                 self._process_data()

    def _process_data(self):
        processed_list = []
        # Consider filtering image_data ONLY if it's not None
        valid_image_keys = []
        if self.image_data:
             valid_image_keys = [
                 k for k, v in self.image_data.items() if v.get('line_info') and isinstance(v['line_info'], list)
             ]
        else: # If image_data is None (e.g., loading precomputed failed and no fallback)
            logging.error("Dataset: Cannot process data, image_data is None.")
            self.processed_data = []
            return

        if not valid_image_keys:
             logging.warning("Dataset: Input image_data contains no samples with valid 'line_info'. Dataset will be empty.")
             self.processed_data = []
             return

        # --- (Processing loop - same as before) ---
        for image_key in tqdm(valid_image_keys, desc="Processing Images"):
            image_sample = self.image_data[image_key]
            line_info_list = image_sample.get('line_info')
            if not line_info_list: continue

            node_features, node_labels, valid_node_indices = [], [], []
            for line_idx, line_dict in enumerate(line_info_list):
                features_tensor = line_dict.get('features')
                if features_tensor is None: continue
                if not isinstance(features_tensor, (torch.Tensor, np.ndarray)): continue
                if isinstance(features_tensor, np.ndarray): features_tensor = torch.from_numpy(features_tensor)
                if features_tensor.shape != (3, 3, 64, 64):
                    print("Incorrect features_tensor.shape detected for line number {}. Skipping this line.".format(line_idx))
                    continue # Skip incorrect shapes

                node_feat = features_tensor.reshape(-1).float() # Flatten

                score = line_dict.get('score')
                if score is None: continue
                try: node_score = float(score)
                except (ValueError, TypeError): continue

                node_features.append(node_feat)
                node_labels.append(node_score)
                valid_node_indices.append(line_idx)

            if not node_features: continue

            x = torch.stack(node_features, dim=0)
            y = torch.tensor(node_labels, dtype=torch.float).unsqueeze(1)
            num_nodes = x.size(0)

            if num_nodes > 1: # Fully connected edge_index
                 adj = torch.ones((num_nodes, num_nodes), dtype=torch.long)
                 adj.fill_diagonal_(0)
                 edge_index, _ = dense_to_sparse(adj)
            else: edge_index = torch.empty((2, 0), dtype=torch.long)

            data = Data(x=x, edge_index=edge_index, y=y, image_key=image_key, num_nodes=num_nodes)
            processed_list.append(data)
        # --- (End Processing loop) ---

        self.processed_data = processed_list
        logging.info(f"Dataset: Processing complete. Created {len(self.processed_data)} graphs.")
        # Automatically save after processing if image_data was provided
        if self.image_data and len(self.processed_data) > 0:
             self.save_dataset() # Save the newly processed data

    def save_dataset(self, path=None):
        save_path = path if path else self._processed_file_path
        save_dir = os.path.dirname(save_path)
        os.makedirs(save_dir, exist_ok=True)
        try:
            with open(save_path, 'wb') as f:
                 pickle.dump(self.processed_data, f)
            logging.info(f"Dataset: Successfully saved {len(self.processed_data)} graphs to: {save_path}")
        except Exception as e:
            logging.error(f"Dataset: Failed to save dataset to {save_path}: {e}")

    def load_dataset(self, path=None):
        load_path = path if path else self._processed_file_path
        if not os.path.exists(load_path):
            logging.error(f"Dataset: Cannot load dataset. File not found: {load_path}")
            self.processed_data = []
            return False # Indicate failure

        try:
            with open(load_path, 'rb') as f:
                loaded_data = pickle.load(f)
            if isinstance(loaded_data, list) and all(isinstance(item, Data) for item in loaded_data):
                self.processed_data = loaded_data
                logging.info(f"Dataset: Successfully loaded {len(self.processed_data)} graphs from: {load_path}")
                return True # Indicate success
            else:
                 logging.error(f"Dataset: Loaded data from {load_path} is not a list of torch_geometric.data.Data objects.")
                 self.processed_data = []
                 return False # Indicate failure
        except Exception as e:
            logging.error(f"Dataset: Failed to load dataset from {load_path}: {e}")
            self.processed_data = []
            return False # Indicate failure

    def len(self): return len(self.processed_data)
    def get(self, idx):
        if not self.processed_data or idx < 0 or idx >= len(self.processed_data):
             # Proper error handling depends on context, raising IndexError is common for get
             raise IndexError(f"Index {idx} out of bounds for dataset with length {len(self.processed_data)}")
        return self.processed_data[idx]

    @property
    def raw_file_names(self): return []
    @property
    def processed_file_names(self): return [os.path.basename(self._processed_file_path)]
# --- End of LineGraphRegressionDatasetInMemory class ---


class LineGraphDataModule(pl.LightningDataModule):
    """
    LightningDataModule for the Line Graph Regression task.

    Uses LineGraphRegressionDatasetInMemory to handle data processing,
    saving, and loading. Manages train/validation/test splits and DataLoaders.
    """
    def __init__(self,
                 image_data: dict,
                 root: str = './graph_data_processed',
                 load_precomputed_dataset: bool = False,
                 batch_size: int = 32,
                 num_workers: int = 0,
                 train_val_test_split: tuple = (0.8, 0.1, 0.1),
                 seed: int = 42):
        """
        Args:
            image_data (dict): Raw image data dictionary (required if not loading precomputed).
            root (str): Directory to save/load the processed graph data file ('processed_graphs.pkl').
            load_precomputed_dataset (bool): If True, attempts to load data from 'root'.
                                             If False, processes 'image_data' unless a
                                             processed file already exists in 'root'.
            batch_size (int): Batch size for DataLoaders.
            num_workers (int): Number of workers for DataLoaders.
            train_val_test_split (tuple): Tuple containing the fractions for
                                          train, validation, and test splits.
                                          Must sum to 1.0. Test split is optional
                                          (e.g., (0.8, 0.2)).
            seed (int): Random seed for reproducible splits.
        """
        super().__init__()
        # Save parameters accessible via self.hparams
        # Note: image_data can be large, consider omitting it from hparams if logging full hyperparameters
        self.save_hyperparameters(ignore=['image_data'])

        self.image_data = image_data # Store raw data reference
        self.root = root
        self.load_precomputed_dataset_flag = load_precomputed_dataset # Renamed to avoid potential hparams conflict if kept
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.train_val_test_split = train_val_test_split
        self.seed = seed

        # Placeholders for datasets after setup
        self.dataset = None
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

    def prepare_data(self):
        """
        Handles dataset processing or checking for precomputed file existence.
        This method is called once per node. Assign no state here (self.xyz = ...).
        It ensures that the data file will be available for `setup`.
        """
        logging.info("DataModule: prepare_data() called.")
        # Instantiate the dataset. Its __init__ handles processing vs. loading logic.
        # We don't store this instance (`_`) long-term here.
        _ = LineGraphRegressionDataset(
            root=self.root,
            image_data=self.image_data,
            load_precomputed_dataset=self.load_precomputed_dataset_flag
        )
        logging.info("DataModule: prepare_data() finished.")
        # Now, either the dataset was loaded, or it was processed (and saved if processing happened).

    def setup(self, stage: str = None):
        """
        Handles dataset loading, splitting, and assignment.
        Called on every GPU process.
        """
        logging.info(f"DataModule: setup(stage={stage}) called.")
        # Instantiate the dataset *again*. Now it should load quickly if prepare_data worked.
        # This ensures each process/GPU gets its own dataset instance correctly.
        self.dataset = LineGraphRegressionDataset(
            root=self.root,
            image_data=self.image_data, # Pass again, though likely ignored if loading
            load_precomputed_dataset=True # Force loading now, prepare_data ensured file exists
        )

        dataset_size = len(self.dataset)
        if dataset_size == 0:
            logging.error("DataModule: Dataset is empty after setup. Cannot create splits.")
            # Set splits to empty lists or None to avoid errors in dataloaders
            self.train_dataset, self.val_dataset, self.test_dataset = [], [], []
            return

        logging.info(f"DataModule: Full dataset size: {dataset_size}")

        # Calculate split lengths
        train_frac, val_frac = self.train_val_test_split[0], self.train_val_test_split[1]
        test_frac = self.train_val_test_split[2] if len(self.train_val_test_split) == 3 else 0.0

        if not np.isclose(train_frac + val_frac + test_frac, 1.0):
             logging.warning(f"Split fractions {self.train_val_test_split} do not sum to 1. Adjusting.")
             # Normalize or raise error - let's normalize test fraction implicitly
             if train_frac + val_frac > 1.0:
                 raise ValueError("Sum of train and validation fractions exceeds 1.0")
             test_frac = 1.0 - train_frac - val_frac

        n_train = int(np.floor(train_frac * dataset_size))
        n_val = int(np.floor(val_frac * dataset_size))
        n_test = dataset_size - n_train - n_val # Ensure all samples are used

        if n_train == 0 or n_val == 0 or (test_frac > 0 and n_test == 0) :
             logging.warning(f"Dataset size ({dataset_size}) too small for requested splits ({n_train}/{n_val}/{n_test}). Some splits might be empty.")

        logging.info(f"DataModule: Splitting into Train: {n_train}, Val: {n_val}, Test: {n_test}")

        # Perform the split using torch.utils.data.random_split
        # Important: random_split works directly on datasets implementing __len__ and __getitem__
        generator = torch.Generator().manual_seed(self.seed)
        splits = random_split(self.dataset, [n_train, n_val, n_test], generator=generator)

        self.train_dataset = splits[0]
        self.val_dataset = splits[1]
        # Only assign test_dataset if test split is non-zero
        self.test_dataset = splits[2] if n_test > 0 else None

        logging.info(f"DataModule: setup() finished. Train size: {len(self.train_dataset)}, Val size: {len(self.val_dataset)}, Test size: {len(self.test_dataset) if self.test_dataset else 0}")


    def train_dataloader(self):
        """Creates the DataLoader for the training set."""
        if not self.train_dataset: return None # Handle empty dataset case
        return DataLoader(self.train_dataset,
                          batch_size=self.batch_size,
                          shuffle=True,
                          num_workers=self.num_workers,
                          pin_memory=True, # Often good for GPU training
                          persistent_workers=True if self.num_workers > 0 else False)

    def val_dataloader(self):
        """Creates the DataLoader for the validation set."""
        if not self.val_dataset: return None
        return DataLoader(self.val_dataset,
                          batch_size=self.batch_size,
                          shuffle=False,
                          num_workers=self.num_workers,
                          pin_memory=True,
                          persistent_workers=True if self.num_workers > 0 else False)

    def test_dataloader(self):
        """Creates the DataLoader for the test set."""
        if not self.test_dataset: return None
        return DataLoader(self.test_dataset,
                          batch_size=self.batch_size,
                          shuffle=False,
                          num_workers=self.num_workers,
                          pin_memory=True,
                          persistent_workers=True if self.num_workers > 0 else False)

    def save_dataset(self, path=None):
        """
        Saves the processed dataset using the underlying dataset's save method.
        Note: This saves the *full* dataset, not the splits.
              Requires `setup()` to have been called at least once to ensure
              `self.dataset` is initialized.

        Args:
            path (str, optional): Path to save the dataset file. Defaults to
                                  the path configured in the underlying dataset.
        """
        if self.dataset is None:
             # Attempt to initialize the dataset if setup hasn't run
             logging.warning("DataModule: save_dataset called before setup. Attempting to initialize dataset first.")
             try:
                 # Initialize just to get access to the save method and path logic
                 # Use load_precomputed=True to avoid reprocessing if possible
                  temp_dataset = LineGraphRegressionDataset(
                     root=self.root, image_data=self.image_data, load_precomputed_dataset=True
                 )
                  temp_dataset.save_dataset(path)
             except Exception as e:
                 logging.error(f"DataModule: Failed to initialize dataset for saving: {e}")
                 print("DataModule: Could not save dataset. Ensure data exists or can be processed.")
        else:
            self.dataset.save_dataset(path)

    def load_dataset(self, path=None):
        """
        Loads the processed dataset using the underlying dataset's load method.
        Note: This loads the *full* dataset. It's generally recommended to use
              the `load_precomputed_dataset=True` flag during initialization.
              Calling this manually *after* setup might require re-running setup
              or the Trainer's fit loop to use the newly loaded data.

        Args:
            path (str, optional): Path to load the dataset file from. Defaults to
                                  the path configured in the underlying dataset.

        Returns:
            bool: True if loading was successful, False otherwise.
        """
        if self.dataset is None:
             # Initialize a temporary dataset instance just to call load
             logging.warning("DataModule: load_dataset called before setup. Attempting to initialize and load.")
             try:
                 temp_dataset = LineGraphRegressionDataset(
                     root=self.root, image_data=None, load_precomputed_dataset=False # Don't trigger processing here
                 )
                 return temp_dataset.load_dataset(path)
             except Exception as e:
                 logging.error(f"DataModule: Failed to initialize dataset for loading: {e}")
                 return False
        else:
            # If setup has run, load into the existing dataset instance
            success = self.dataset.load_dataset(path)
            if success:
                logging.info("DataModule: Dataset loaded successfully via load_dataset(). You may need to re-run setup or trainer.fit() to apply changes.")
            return success


# # 1. Make sure you have your 'image_data' dictionary populated
# # Example (replace with your actual data loading):
# # image_data = load_my_data_function(...)

# # Check if image_data exists and is a dictionary
# if 'image_data' in locals() and isinstance(image_data, dict):
#
#     # 2. Define the root directory for processed data
#     dataset_root = './my_graph_dataset'
#     os.makedirs(dataset_root, exist_ok=True) # Ensure directory exists
#
#     # 3. Instantiate the dataset
#     # This will automatically trigger the .process() method if
#     # the processed files don't exist in dataset_root/processed/
#     print(f"Initializing dataset. Processing data if needed...")
#     dataset = LineGraphRegressionDataset(root=dataset_root, image_data=image_data)
#     print(f"Dataset ready. Number of graphs: {len(dataset)}")
#
#     # 4. Access data samples (optional)
#     if len(dataset) > 0:
#         first_graph = dataset[0]
#         print("\n--- First Graph Sample ---")
#         print(first_graph)
#         print(f"Number of nodes: {first_graph.num_nodes}")
#         print(f"Node features shape: {first_graph.x.shape}")
#         print(f"Node labels shape: {first_graph.y.shape}")
#         print(f"Original image key: {first_graph.image_key}") # Example of accessing custom attribute
#
#     # 5. Use with DataLoader (example)
#     from torch_geometric.loader import DataLoader
#     loader = DataLoader(dataset, batch_size=4, shuffle=True)
#
#     print("\n--- DataLoader Example ---")
#     # Iterate through batches
#     # for batch in loader:
#     #    print(f"Processing batch with {batch.num_graphs} graphs...")
#     #    # Your training/evaluation logic here using 'batch'
#     #    pass
#
# else:
#     print("Error: The 'image_data' variable is not defined or is not a dictionary.")
#     print("Please ensure your data is loaded into a dictionary named 'image_data' before instantiating the dataset.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from torch_geometric.nn import GATConv, LayerNorm # Import LayerNorm
import wandb
from pytorch_lightning.callbacks import ModelCheckpoint
import numpy as np
from torch_geometric.data import Dataset, Data
from torch_geometric.utils import dense_to_sparse
from torch_geometric.loader import DataLoader # Use PyG DataLoader
from torch.utils.data import random_split, Subset # For splitting
from tqdm import tqdm
import logging
import pickle
import os # Make sure os is imported


class CNNFeatureExtractor(nn.Module):
    def __init__(self, output_dim=128):
        super().__init__()
        # Assuming input is conceptually (N, 9, 64, 64) where 9 = 3x3 channels
        # Or handle the (3, 3, 64, 64) shape inside forward if needed
        self.conv1 = nn.Conv2d(9, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # -> 32x32
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # -> 16x16
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2) # -> 8x8
        # Global Average Pooling or Flatten + Linear
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, output_dim)
        
    def forward(self, x):
        # x likely has shape [num_nodes_in_batch, 3, 3, 64, 64]
        # Reshape for Conv2d: [num_nodes_in_batch, 9, 64, 64]
        num_nodes = x.shape[0]
        x = x.view(num_nodes, 9, 64, 64)

        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = self.adaptive_pool(x) # -> [num_nodes, 128, 1, 1]
        x = torch.flatten(x, 1)   # -> [num_nodes, 128]
        x = F.relu(self.fc(x))    # -> [num_nodes, output_dim]
        return x




# --- Dataset Class (Minimal changes for clarity/robustness) ---
# Configure logging (ensure it's configured globally or within your main script)
# logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # Uncomment if needed

class LineGraphRegressionDataset(Dataset):
    """
    PyTorch Geometric Dataset for graph regression on line features.
    (Keeps data in memory, handles processing/saving/loading)
    """
    def __init__(self, root, image_data, load_precomputed_dataset=False):
        self.root = root
        self.image_data = image_data # Keep reference if needed for reprocessing
        self._processed_file_path = os.path.join(self.root, 'processed_graphs.pkl')
        self.processed_data = []
        # Initialize first, then decide action based on flags/files
        super().__init__(root) # Call super().__init__ after self.root is set
        
        if load_precomputed_dataset:
            if os.path.exists(self._processed_file_path):
                logging.info(f"Dataset: Attempting to load precomputed data from: {self._processed_file_path}")
                loaded = self.load_dataset(self._processed_file_path)
                if not loaded: # If loading failed, process from image_data if available
                    logging.warning("Dataset: Loading failed. Processing from image_data instead.")
                    if self.image_data:
                         self._process_data()
                    else:
                         logging.error("Dataset: Cannot process data after failed load, image_data is None.")
            else:
                logging.warning(f"Dataset: load_precomputed_dataset=True but file not found at: {self._processed_file_path}. Processing from image_data if available.")
                if self.image_data:
                     self._process_data()
                else:
                     logging.error("Dataset: Cannot process data, image_data is None.")
        else:
            # If not loading precomputed, always process (overwrite if exists)
            logging.info("Dataset: load_precomputed_dataset=False. Processing dataset from provided image_data...")
            if self.image_data:
                self._process_data()
            else:
                logging.error("Dataset: Cannot process data, image_data is None.")


    def _process_data(self):
        processed_list = []
        valid_image_keys = []
        if self.image_data:
             valid_image_keys = [
                 k for k, v in self.image_data.items() if v.get('line_info') and isinstance(v['line_info'], list)
             ]
        else:
            logging.error("Dataset: Cannot process data, image_data is None.")
            self.processed_data = []
            return

        if not valid_image_keys:
             logging.warning("Dataset: Input image_data contains no samples with valid 'line_info'. Dataset will be empty.")
             self.processed_data = []
             return

        for image_key in tqdm(valid_image_keys, desc="Processing Images"):
            image_sample = self.image_data[image_key]
            line_info_list = image_sample.get('line_info')
            if not line_info_list: continue

            node_features, node_labels, valid_node_indices = [], [], []
            for line_idx, line_dict in enumerate(line_info_list):
                features_tensor = line_dict.get('features')
                if features_tensor is None: continue
                if not isinstance(features_tensor, (torch.Tensor, np.ndarray)): continue
                if isinstance(features_tensor, np.ndarray): features_tensor = torch.from_numpy(features_tensor)
                if features_tensor.shape != (3, 3, 64, 64):
                    # Use logging instead of print
                    logging.warning(f"Incorrect features_tensor.shape {features_tensor.shape} detected for image {image_key}, line index {line_idx}. Expected (3, 3, 64, 64). Skipping line.")
                    continue # Skip incorrect shapes

                node_feat = features_tensor.reshape(-1).float() # Flatten

                score = line_dict.get('score')
                if score is None: continue
                try:
                    node_score = float(score)
                    # Add check/warning if score is outside expected [0, 1] range for Sigmoid
                    if not (0.0 <= node_score <= 1.0):
                        logging.warning(f"Score {node_score} for image {image_key}, line {line_idx} is outside [0, 1] range. Sigmoid output assumes [0, 1].")
                except (ValueError, TypeError):
                    logging.warning(f"Invalid score type {type(score)} or value for image {image_key}, line {line_idx}. Skipping line.")
                    continue

                node_features.append(node_feat)
                node_labels.append(node_score)
                valid_node_indices.append(line_idx)

            if not node_features: continue

            x = torch.stack(node_features, dim=0)
            y = torch.tensor(node_labels, dtype=torch.float).unsqueeze(1)
            num_nodes = x.size(0)

            if num_nodes > 1: # Fully connected edge_index
                 adj = torch.ones((num_nodes, num_nodes), dtype=torch.long)
                 adj.fill_diagonal_(0)
                 edge_index, _ = dense_to_sparse(adj)
            else: edge_index = torch.empty((2, 0), dtype=torch.long)

            # Ensure data consistency
            if y.shape[0] != num_nodes:
                logging.error(f"Shape mismatch in image {image_key}: num_nodes={num_nodes}, y.shape={y.shape}. Skipping graph.")
                continue

            data = Data(x=x, edge_index=edge_index, y=y, image_key=image_key, num_nodes=num_nodes)
            processed_list.append(data)

        self.processed_data = processed_list
        logging.info(f"Dataset: Processing complete. Created {len(self.processed_data)} graphs.")
        if len(self.processed_data) > 0:
             self.save_dataset() # Save the newly processed data

    def save_dataset(self, path=None):
        save_path = path if path else self._processed_file_path
        save_dir = os.path.dirname(save_path)
        os.makedirs(save_dir, exist_ok=True)
        try:
            with open(save_path, 'wb') as f:
                 pickle.dump(self.processed_data, f)
            logging.info(f"Dataset: Successfully saved {len(self.processed_data)} graphs to: {save_path}")
        except Exception as e:
            logging.error(f"Dataset: Failed to save dataset to {save_path}: {e}")

    def load_dataset(self, path=None):
        load_path = path if path else self._processed_file_path
        if not os.path.exists(load_path):
            logging.error(f"Dataset: Cannot load dataset. File not found: {load_path}")
            self.processed_data = []
            return False

        try:
            with open(load_path, 'rb') as f:
                loaded_data = pickle.load(f)
            if isinstance(loaded_data, list) and all(isinstance(item, Data) for item in loaded_data):
                # Add basic validation check on loaded data
                if not loaded_data:
                    logging.warning(f"Dataset: Loaded file {load_path} contains an empty list.")
                    self.processed_data = []
                    return True # File loaded, but it's empty

                first_item = loaded_data[0]
                if not hasattr(first_item, 'x') or not hasattr(first_item, 'edge_index') or not hasattr(first_item, 'y'):
                     logging.error(f"Dataset: Data in {load_path} seems incomplete (missing x, edge_index, or y).")
                     self.processed_data = []
                     return False

                self.processed_data = loaded_data
                logging.info(f"Dataset: Successfully loaded {len(self.processed_data)} graphs from: {load_path}")
                return True
            else:
                 logging.error(f"Dataset: Loaded data from {load_path} is not a list of torch_geometric.data.Data objects.")
                 self.processed_data = []
                 return False
        except (pickle.UnpicklingError, EOFError, AttributeError, ImportError, IndexError) as e:
            logging.error(f"Dataset: Failed to load or parse dataset from {load_path} (file might be corrupted or incompatible): {e}")
            self.processed_data = []
            return False
        except Exception as e:
            logging.error(f"Dataset: An unexpected error occurred loading dataset from {load_path}: {e}")
            self.processed_data = []
            return False

    def len(self): return len(self.processed_data)
    def get(self, idx):
        if not self.processed_data or idx < 0 or idx >= len(self.processed_data):
             raise IndexError(f"Index {idx} out of bounds for dataset with length {len(self.processed_data)}")
        return self.processed_data[idx]

    @property
    def raw_file_names(self): return []
    @property
    def processed_file_names(self): return [os.path.basename(self._processed_file_path)] if hasattr(self, '_processed_file_path') else []

# --- Data Module Class (Mostly unchanged, relies on Dataset's logic) ---
class LineGraphDataModule(pl.LightningDataModule):
    def __init__(self,
                 image_data: dict, # Can be None if load_precomputed_dataset is True and file exists
                 root: str = './graph_data_processed',
                 load_precomputed_dataset: bool = False,
                 batch_size: int = 32,
                 num_workers: int = 0,
                 train_val_test_split: tuple = (0.8, 0.1, 0.1),
                 seed: int = 42):
        super().__init__()
        # image_data can be large, avoid saving to hparams log file
        self.save_hyperparameters(ignore=['image_data'])

        # Store args directly for internal use
        self._image_data = image_data
        self._root = root
        self._load_precomputed_dataset_flag = load_precomputed_dataset
        self._batch_size = batch_size
        self._num_workers = num_workers
        self._train_val_test_split = train_val_test_split
        self._seed = seed

        self.dataset = None
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

    def prepare_data(self):
        logging.info("DataModule: prepare_data() called.")
        # Instantiate the dataset; its __init__ handles processing/loading logic.
        # We don't need to store the instance ('_') here.
        _ = LineGraphRegressionDataset(
            root=self._root,
            image_data=self._image_data,
            load_precomputed_dataset=self._load_precomputed_dataset_flag
        )
        # After this, the processed file should exist (either loaded or created).
        logging.info("DataModule: prepare_data() finished.")


    def setup(self, stage: str = None):
        logging.info(f"DataModule: setup(stage={stage}) called.")
        # Instantiate dataset *again*. This time, force loading as prepare_data should
        # have ensured the file exists. Pass image_data=None as it's not needed for loading.
        self.dataset = LineGraphRegressionDataset(
            root=self._root,
            image_data=None, # Not needed if loading
            load_precomputed_dataset=True # Force loading attempt
        )

        dataset_size = len(self.dataset)
        if dataset_size == 0:
            logging.error("DataModule: Dataset is empty after setup. Cannot create splits.")
            self.train_dataset, self.val_dataset, self.test_dataset = [], [], []
            return

        logging.info(f"DataModule: Full dataset size: {dataset_size}")

        # Calculate split lengths
        if len(self._train_val_test_split) == 2: # Handle (train, val) tuple
             train_frac, val_frac = self._train_val_test_split
             test_frac = 0.0
        elif len(self._train_val_test_split) == 3:
             train_frac, val_frac, test_frac = self._train_val_test_split
        else:
            raise ValueError("train_val_test_split must be a tuple of 2 or 3 floats.")


        if not np.isclose(train_frac + val_frac + test_frac, 1.0):
             logging.warning(f"Split fractions {self._train_val_test_split} do not sum to 1. Normalizing.")
             total_frac = train_frac + val_frac + test_frac
             train_frac /= total_frac
             val_frac /= total_frac
             test_frac /= total_frac


        n_train = int(np.floor(train_frac * dataset_size))
        n_val = int(np.floor(val_frac * dataset_size))
        # Assign remaining to test split to ensure all data is used
        n_test = dataset_size - n_train - n_val

        # Handle cases where dataset is too small for splits
        if dataset_size > 0 and (n_train == 0 or n_val == 0):
             logging.warning(f"Dataset size ({dataset_size}) is small for the split ratios. Train ({n_train})/Val ({n_val}) splits might be empty or very small.")
        if test_frac > 0 and n_test == 0 and dataset_size > 0:
             logging.warning(f"Test split fraction requested, but calculated size is 0 ({n_test}).")


        logging.info(f"DataModule: Splitting into Train: {n_train}, Val: {n_val}, Test: {n_test}")

        # Perform split
        generator = torch.Generator().manual_seed(self._seed)
        # Ensure split lengths sum exactly to dataset_size
        split_lengths = [n_train, n_val, n_test]
        if sum(split_lengths) != dataset_size:
             # Adjust the largest split (usually train) if there's a rounding discrepancy
             diff = dataset_size - sum(split_lengths)
             split_lengths[0] += diff
             logging.info(f"Adjusting split lengths slightly due to rounding: {split_lengths}")


        # Check if any split length is negative (shouldn't happen with floor and remainder logic)
        if any(s < 0 for s in split_lengths):
             raise ValueError(f"Calculated negative split size: {split_lengths}. Check ratios and dataset size.")


        # Only split if dataset is not empty
        if dataset_size > 0 :
            try:
                splits = random_split(self.dataset, split_lengths, generator=generator)
                self.train_dataset = splits[0]
                self.val_dataset = splits[1]
                self.test_dataset = splits[2] if n_test > 0 else None # Assign None if test size is 0
            except ValueError as e:
                 logging.error(f"Error during random_split: {e}. Dataset size: {dataset_size}, split lengths: {split_lengths}")
                 # Set splits to empty to prevent dataloader errors
                 self.train_dataset, self.val_dataset, self.test_dataset = [], [], []
        else:
            self.train_dataset, self.val_dataset, self.test_dataset = [], [], []


        logging.info(f"DataModule: setup() finished. Train size: {len(self.train_dataset)}, Val size: {len(self.val_dataset)}, Test size: {len(self.test_dataset) if self.test_dataset else 0}")


    def train_dataloader(self):
        if not self.train_dataset: return None
        return DataLoader(self.train_dataset,
                          batch_size=self._batch_size,
                          shuffle=True,
                          num_workers=self._num_workers,
                          pin_memory=True,
                          persistent_workers=True if self._num_workers > 0 else False)

    def val_dataloader(self):
        if not self.val_dataset: return None
        return DataLoader(self.val_dataset,
                          batch_size=self._batch_size,
                          shuffle=False,
                          num_workers=self._num_workers,
                          pin_memory=True,
                          persistent_workers=True if self._num_workers > 0 else False)

    def test_dataloader(self):
        if not self.test_dataset: return None
        return DataLoader(self.test_dataset,
                          batch_size=self._batch_size,
                          shuffle=False,
                          num_workers=self._num_workers,
                          pin_memory=True,
                          persistent_workers=True if self._num_workers > 0 else False)

    # save/load methods delegate to the underlying dataset instance
    def save_dataset(self, path=None):
        if self.dataset is None:
            logging.warning("DataModule: save_dataset called before setup. Dataset not initialized.")
            # Optionally, try to initialize and save if needed, but it's better practice to ensure setup is called first.
            # self.setup() # Could call setup, but might have side effects
            print("DataModule: Cannot save dataset. Run setup() first.")
            return
        self.dataset.save_dataset(path)

    def load_dataset(self, path=None):
         if self.dataset is None:
             logging.warning("DataModule: load_dataset called before setup. Initializing temporary dataset for loading.")
             # Create a temporary instance just to load
             temp_dataset = LineGraphRegressionDataset(root=self._root, image_data=None, load_precomputed_dataset=False)
             return temp_dataset.load_dataset(path)
             # Note: This loaded data isn't automatically used unless setup is run again.
         else:
             # Load into the existing dataset instance
             success = self.dataset.load_dataset(path)
             if success:
                 logging.info("DataModule: Dataset loaded successfully via load_dataset(). Re-run setup() or trainer.fit() to use the new data.")
             return success


# --- GAT Regressor Model (Added LayerNorm,) ---
class GATRegressor(pl.LightningModule):
    def __init__(self,
                 input_dim: int,
                 cnn_output_dim = 128,
                 hidden_dim: int = 128,
                 output_dim: int = 1,
                 n_heads: int = 4,
                 n_layers: int = 2,
                 dropout: float = 0.2,
                 learning_rate: float = 5e-4):
        super().__init__()
        self.save_hyperparameters()
        self.feature_extractor = CNNFeatureExtractor(output_dim=cnn_output_dim)
        # --- Network Architecture ---
        self.input_embed = nn.Linear(cnn_output_dim, self.hparams.hidden_dim)
        # Add LayerNorm after initial embedding
        self.input_norm = LayerNorm(self.hparams.hidden_dim)

        self.gat_layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList() # Store LayerNorm layers corresponding to GAT layers
        current_dim = self.hparams.hidden_dim

        for i in range(self.hparams.n_layers):
            is_last_layer = (i == self.hparams.n_layers - 1)
            heads = 1 if is_last_layer else self.hparams.n_heads
            concat = False if is_last_layer else True
            gat_input_dim = current_dim # Input to GAT is output of previous layer (or input_embed)

            conv = GATConv(gat_input_dim,
                           self.hparams.hidden_dim,
                           heads=heads,
                           dropout=self.hparams.dropout,
                           concat=concat)
            self.gat_layers.append(conv)

            # Determine the output dimension of the GAT layer
            if concat:
                 current_dim = self.hparams.hidden_dim * heads
            else: # Last layer with concat=False
                 current_dim = self.hparams.hidden_dim

            # Add LayerNorm after each GAT layer's activation/dropout (except maybe the very last)
            if not is_last_layer: # Don't normalize right before the final linear output layer
                 self.norm_layers.append(LayerNorm(current_dim))
            # Note: current_dim now reflects the output dimension for the *next* layer's input or the final output layer

        # Final Output Layer
        # Input dim is the output dim of the last GAT layer
        final_gat_output_dim = self.hparams.hidden_dim # Since last layer has concat=False, heads=1
        self.output_layer = nn.Linear(final_gat_output_dim, self.hparams.output_dim)

        self.loss_fn = nn.BCEWithLogitsLoss()
        

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        # 0. Apply CNN
        x = self.feature_extractor(x)
        # 1. Apply further linear layer, normalization, activation, dropout
        x = self.input_embed(x)
        x = self.input_norm(x) # Apply LayerNorm
        x = F.relu(x)
        x = F.dropout(x, p=self.hparams.dropout, training=self.training)

        # 2. GAT layers with intermediate normalization
        for i, layer in enumerate(self.gat_layers):
            x = layer(x, edge_index)
            is_last_layer = (i == self.hparams.n_layers - 1)

            if not is_last_layer:
                # Apply norm -> activation -> dropout for intermediate layers
                x = self.norm_layers[i](x) # Apply the corresponding LayerNorm
                x = F.relu(x)
                x = F.dropout(x, p=self.hparams.dropout, training=self.training)
            # No activation/dropout/norm after the final GAT layer before the output layer

        # 3. Final output layer
        logits= self.output_layer(x)


        return logits

    def _calculate_accuracy(self, y_hat, y, batch_vector):
        # Ensure tensors are on the same device for comparison
        y = y.to(y_hat.device)
        batch_vector = batch_vector.to(y_hat.device)

        # Use a threshold (e.g., 0.5 if using sigmoid) for accuracy
        # Adjust threshold if needed, or use a different metric (e.g., MAE)
        y_true_thresh = (y >= 0.5).float()
        y_pred_thresh = (y_hat >= 0.5).float()

        accuracies = []
        for graph_idx in torch.unique(batch_vector):
            mask = (batch_vector == graph_idx)
            nodes_in_graph = mask.sum().item()

            if nodes_in_graph > 0:
                # Compare thresholded values
                correct_predictions = (y_true_thresh[mask] == y_pred_thresh[mask]).sum().item()
                graph_accuracy = correct_predictions / nodes_in_graph
                accuracies.append(graph_accuracy)

        if not accuracies:
            return torch.tensor(0.0, device=self.device)

        avg_accuracy = torch.tensor(accuracies, device=self.device).mean()
        return avg_accuracy


    def _shared_step(self, batch, batch_idx):
        if not hasattr(batch, 'y') or not hasattr(batch, 'batch'):
            # Check if batch size is 0, can happen if last batch is skipped or dataset empty
            if batch.num_graphs == 0:
                 logging.warning(f"Skipping step {batch_idx}: Batch contains 0 graphs.")
                 # Return None or dummy values to avoid errors downstream
                 # Returning None might require handling in the Pytorch Lightning training loop internals
                 # Let's return zero loss/acc for now, but this batch contributes nothing.
                 return torch.tensor(0.0, device=self.device, requires_grad=True), torch.tensor(0.0, device=self.device) # Loss needs grad potentially
            else:
                raise ValueError("Batch object must have 'y' (labels) and 'batch' attributes.")

        y_hat = self.forward(batch) # [N, 1] or [N] if output_dim=1 was squeezed earlier

        # Ensure y exists and has data
        if batch.y is None or batch.y.numel() == 0:
             logging.warning(f"Skipping step {batch_idx}: Batch has missing or empty labels 'y'.")
             return torch.tensor(0.0, device=self.device, requires_grad=True), torch.tensor(0.0, device=self.device)


        # --- Shape Handling ---
        # Squeeze prediction if output_dim is 1
        if self.hparams.output_dim == 1 and y_hat.dim() > 1:
            y_hat = y_hat.squeeze(-1) # -> [N]

        # Ensure y is also squeezed if it's [N, 1]
        y = batch.y
        if y.dim() > 1 and y.shape[1] == 1:
            y = y.squeeze(-1) # -> [N]

        # Final shape check
        if y_hat.shape != y.shape:
             # Check if the mismatch is due to an empty graph processed somehow
             if y_hat.numel() == 0 and y.numel() == 0:
                  logging.warning(f"Step {batch_idx}: Both prediction and target are empty (likely empty graph). Returning zero loss.")
                  return torch.tensor(0.0, device=self.device, requires_grad=True), torch.tensor(0.0, device=self.device)
             else:
                 raise RuntimeError(f"Shape mismatch before loss calculation: y_hat {y_hat.shape}, y {y.shape} in batch {batch_idx}")

        # Check for NaNs/Infs *before* loss calculation
        if torch.isnan(y_hat).any() or torch.isinf(y_hat).any():
            logging.error(f"NaN or Inf detected in model output (y_hat) at step {batch_idx}!")
            # Optionally: return a large loss or handle differently
            # For now, let loss calculation proceed, which will likely result in NaN loss
            pass
        if torch.isnan(y).any() or torch.isinf(y).any():
            logging.error(f"NaN or Inf detected in labels (y) at step {batch_idx}!")
             # Return zero loss/acc if labels are bad
            return torch.tensor(0.0, device=self.device, requires_grad=True), torch.tensor(0.0, device=self.device)


        # --- Loss and Accuracy ---
        loss = self.loss_fn(y_hat, y)

        # Check for NaN loss
        if torch.isnan(loss):
            logging.error(f"NaN loss detected at step {batch_idx}! y_hat min/max: {y_hat.min()}/{y_hat.max()}, y min/max: {y.min()}/{y.max()}")
            # Consider alternatives: return 0 loss, raise error, or try to debug further
            # Returning 0 might mask the problem, but prevents crashing training immediately
            # loss = torch.tensor(0.0, device=self.device, requires_grad=True) # Example: replace NaN loss

        # Calculate accuracy only if loss is valid
        accuracy = torch.tensor(0.0, device=self.device) # Default
        if not torch.isnan(loss) and not torch.isinf(loss):
             # Ensure batch vector is valid before calculating accuracy
             if hasattr(batch, 'batch') and batch.batch is not None and batch.batch.numel() == y.numel():
                 accuracy = self._calculate_accuracy(y_hat, y, batch.batch)
             else:
                 logging.warning(f"Cannot calculate accuracy at step {batch_idx}: Invalid or mismatched 'batch' vector.")


        return loss, accuracy

    def training_step(self, batch, batch_idx):
        loss, accuracy = self._shared_step(batch, batch_idx)
        # Log metrics, providing batch_size for correct aggregation
        batch_size = batch.num_graphs if hasattr(batch, 'num_graphs') else 0 # Get actual number of graphs
        if batch_size > 0:
             self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True, batch_size=batch_size)
             self.log('train_avg_acc', accuracy, on_step=True, on_epoch=True, prog_bar=False, logger=True, batch_size=batch_size)
        elif loss is not None: # Log step loss even if batch size is 0, but epoch avg will be wrong
             self.log('train_loss_step', loss, on_step=True, on_epoch=False, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, accuracy = self._shared_step(batch, batch_idx)
        batch_size = batch.num_graphs if hasattr(batch, 'num_graphs') else 0
        if batch_size > 0 and loss is not None and not torch.isnan(loss): # Only log valid steps
            self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True, logger=True, batch_size=batch_size)
            self.log('val_avg_acc', accuracy, on_step=False, on_epoch=True, prog_bar=True, logger=True, batch_size=batch_size)
        # No return needed unless you aggregate manually

    def test_step(self, batch, batch_idx):
        loss, accuracy = self._shared_step(batch, batch_idx)
        batch_size = batch.num_graphs if hasattr(batch, 'num_graphs') else 0
        if batch_size > 0 and loss is not None and not torch.isnan(loss): # Only log valid steps
            self.log('test_loss', loss, on_step=False, on_epoch=True, logger=True, batch_size=batch_size)
            self.log('test_avg_acc', accuracy, on_step=False, on_epoch=True, logger=True, batch_size=batch_size)
        # No return needed

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        return optimizer

# --- Main Script Execution ---

# Configure logging (do this once at the start)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 1. Load your image_data (replace with your actual loading)
# Placeholder: Assume image_data is loaded correctly
# Example: image_data = {... your data ...}
# Make sure this variable exists and is populated before proceeding
if 'image_data' not in locals() or not isinstance(image_data, dict):
      logging.error("The 'image_data' dictionary is not defined or populated. Load your data first.")
      # Example dummy data for testing structure (replace!)
      # image_data = {'img1': {'line_info': [{'features': np.random.rand(3, 3, 64, 64).astype(np.float32), 'score': 0.8},
      #                                      {'features': np.random.rand(3, 3, 64, 64).astype(np.float32), 'score': 0.3}]},
      #               'img2': {'line_info': [{'features': np.random.rand(3, 3, 64, 64).astype(np.float32), 'score': 0.9}]}}
      # If you don't have data yet, exit or load dummy data.
      exit() # Or raise error


# --- Dataset Instantiation  ---
dataset_path = 'graph_dataset/' # Or your preferred path
os.makedirs(dataset_path, exist_ok=True)
logging.info(f"Initializing dataset. Processing data if needed...")
# Let's try loading first if the file exists, otherwise process
processed_file = os.path.join(dataset_path, 'processed_graphs.pkl')
should_load_precomputed = os.path.exists(processed_file)
logging.info(f"Precomputed file exists: {should_load_precomputed}. Setting load_precomputed_dataset accordingly.")

# Instantiate dataset (using the variable determined above)
dataset = LineGraphRegressionDataset(root=dataset_path,
                                     image_data=image_data,
                                     load_precomputed_dataset=should_load_precomputed)

if len(dataset) > 0:
    logging.info(f"Dataset ready. Number of graphs: {len(dataset)}")
    first_graph = dataset[0]
    logging.info(f"First graph: Nodes={first_graph.num_nodes}, Features={first_graph.x.shape}, Labels={first_graph.y.shape}, Key={getattr(first_graph, 'image_key', 'N/A')}")
else:
    logging.error("Dataset is empty after initialization. Check data processing or loading.")
    exit()


# --- Data Module Instantiation ---
data_module_root = "/Users/matinurdu/Desktop/ETH_Zürich/FS25/3D-Vision/DeepLSD/notebooks/graph_dataset" # Module's root directory
batch_s = 8 # Keep batch size 1 if higher values caused issues before, but try increasing later
train_val_test_ratio = (0.6, 0.2, 0.2)
# Let DataModule decide whether to load based on file existence via prepare_data
load_precomputed_dm = True # Set to True so setup *attempts* loading first

logging.info(f"\n--- Initializing DataModule ---")
data_module = LineGraphDataModule(
    image_data=image_data, # Pass image data for potential reprocessing if loading fails
    root=data_module_root,
    load_precomputed_dataset=load_precomputed_dm, # Flag for prepare_data
    batch_size=batch_s,
    train_val_test_split=train_val_test_ratio
)

# --- Prepare and Setup Data ---
logging.info(f"\n--- Preparing Data ---")
data_module.prepare_data() # Ensures processed file exists

logging.info(f"\n--- Setting up DataLoaders ---")
data_module.setup() # Loads data and creates splits

# Optional: Save dataset state after setup if needed (usually done by prepare_data if processed)
# data_module.save_dataset()

# --- Verification ---
logging.info(f"\n--- Verifying Train DataLoader ---")
train_loader = data_module.train_dataloader()

if train_loader is not None and len(train_loader) > 0:
    logging.info(f"Number of training batches: {len(train_loader)}")
    try:
        first_batch = next(iter(train_loader))
        logging.info(f"First batch type: {type(first_batch)}")
        if hasattr(first_batch, 'num_graphs'):
            logging.info(f"Number of graphs in first batch: {first_batch.num_graphs}")
            if first_batch.num_graphs > 0:
                first_graph_in_batch = first_batch.get_example(0)
                logging.info("\n--- First Graph in First Training Batch ---")
                logging.info(f"Nodes: {first_graph_in_batch.num_nodes}, Features: {first_graph_in_batch.x.shape}, Labels: {first_graph_in_batch.y.shape}")
                if hasattr(first_graph_in_batch, 'image_key'):
                    logging.info(f"Original image key: {first_graph_in_batch.image_key}")
                logging.info(f"Edge index shape: {first_graph_in_batch.edge_index.shape}")
            else:
                 logging.warning("First batch contains 0 graphs.")
        else:
            logging.warning("First batch object does not have 'num_graphs' attribute.")

    except StopIteration:
        logging.error("Train DataLoader is empty, cannot get first batch.")
    except Exception as e:
        logging.error(f"Error while inspecting first batch: {e}")

elif data_module.train_dataset is not None and len(data_module.train_dataset) == 0:
     logging.warning("Training dataset is empty (likely due to small total dataset size or split configuration).")
else:
    logging.warning("Train DataLoader could not be created or is None.")


# --- Training Setup ---
# Login to Wandb (do this *before* initializing the logger)
try:
    # Use wandb login from CLI or environment variable for API key for better practice
    # wandb.login(key="YOUR_KEY") # Replace with your key if needed, but CLI/env var is preferred
    wandb_logger = pl.loggers.WandbLogger(project="graph-line-regression", log_model="all") # Changed project name slightly
    logging.info("Wandb logger initialized.")
except Exception as e:
    logging.error(f"Failed to initialize Wandb logger: {e}. Training will proceed without Wandb logging.")
    wandb_logger = None # Set logger to None to disable logging


# Check input dimension from the first graph
# It's safer to get this dynamically after data loading
if len(dataset) > 0:
    input_feature_dim = dataset[0].x.shape[1]
    logging.info(f"Determined input feature dimension: {input_feature_dim}")
else:
    logging.error("Cannot determine input feature dimension, dataset is empty.")
    exit()

model = GATRegressor(
        input_dim=input_feature_dim, # Use dynamically determined dim
        cnn_output_dim=128,
        hidden_dim=128,
        n_heads=4,
        n_layers=3,
        dropout=0.2,
        learning_rate=5e-4
    )

checkpoint_callback = ModelCheckpoint(
    dirpath="./checkpoints",
    filename="gat-line-{epoch}-{val_loss:.2f}", # Include val_loss in name
    save_top_k=1,        # Save top 2 models based on monitored metric
    monitor="val_loss",  # Monitor validation loss
    mode="min",          # Save models with minimum validation loss
    save_last=True       # Also save the last epoch's checkpoint
)

trainer = pl.Trainer(
        logger=wandb_logger, # Use the logger instance (can be None)
        max_epochs=30,
        accelerator='cpu', # Automatically choose accelerator (mps, cpu, gpu)
        gradient_clip_val=1.0, # Added gradient clipping
        log_every_n_steps=2, # Log slightly less often
        callbacks=[checkpoint_callback],
        # precision="16-mixed" # Optional: Use mixed precision if running on GPU for speed/memory
        # detect_anomaly=True # Optional: Enable anomaly detection during training for debugging NaNs
    )

# --- Run Training ---
logging.info("Starting training...")
try:
    trainer.fit(model, datamodule=data_module)
    logging.info("Training finished.")
except Exception as e:
    logging.error(f"An error occurred during training: {e}", exc_info=True) # Log traceback


if data_module.test_dataloader() is not None:
    logging.info("Starting testing...")
    try:
        trainer.test(model, datamodule=data_module)
        logging.info("Testing finished.")
    except Exception as e:
        logging.error(f"An error occurred during testing: {e}", exc_info=True)
else:
    logging.info("No test dataset available, skipping testing.")


# --- Finish Wandb ---
if wandb_logger is not None:
    wandb.finish()
    logging.info("Wandb run finished.")

logging.info("Script execution complete.")

In [8]:
# """ # import os
# # os.environ["PYTORCH_ENABLE_MPS_FALLBACK"]="1"

# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import pytorch_lightning as pl
# from torch_geometric.nn import GATConv # Import GAT layer
# import wandb # Make sure wandb is installed: pip install wandb
# from pytorch_lightning.callbacks import ModelCheckpoint


# class GATRegressor(pl.LightningModule):
#     #Graph Attention Network for Node Regression implemented with PyTorch Lightning.
#     #Predicts a score for each node (line) in a graph.
#     def __init__(self,
#                  input_dim: int, # Should be 36864 based on flattened features
#                  hidden_dim: int = 128,
#                  output_dim: int = 1, # Predicting a single score per node
#                  n_heads: int = 4,    # Number of attention heads in GAT layers
#                  n_layers: int = 2,   # Number of GAT layers
#                  dropout: float = 0.2,
#                  learning_rate: float = 1e-3,
#                  batch_size: int = 64):
#         super().__init__()

#         # Store hyperparameters for logging and potential loading
#         self.save_hyperparameters()

#         # --- Network Architecture ---
#         # 1. Initial Embedding Layer to reduce high input dimensionality
#         self.input_embed = nn.Linear(self.hparams.input_dim, self.hparams.hidden_dim)

#         # 2. GAT Layers
#         self.gat_layers = nn.ModuleList()
#         current_dim = self.hparams.hidden_dim

#         for i in range(self.hparams.n_layers):
#             # Last layer: Use 1 head and disable concatenation for final output features
#             is_last_layer = (i == self.hparams.n_layers - 1)
#             heads = 1 if is_last_layer else self.hparams.n_heads
#             concat = False if is_last_layer else True

#             # Input dim for GAT layer needs to account for concatenated heads from prev layer
#             gat_input_dim = current_dim if i == 0 else current_dim * self.hparams.n_heads

#             conv = GATConv(gat_input_dim,
#                            self.hparams.hidden_dim,
#                            heads=heads,
#                            dropout=self.hparams.dropout,
#                            concat=concat) # concat=False only makes sense if heads=1
#             self.gat_layers.append(conv)

#             # Update current_dim for the next layer's input size calculation if needed
#             # If concat=True, output dim is hidden_dim * heads.
#             # If concat=False (last layer), output dim is hidden_dim.
#             # Since the last layer has concat=False, the final GAT output dim is hidden_dim.
#             # Note: This logic assumes n_layers >= 1

#         # 3. Final Output Layer - with Sigmoid to ensure output between 0 and 1
#         # The input dimension depends on the output of the last GAT layer
#         # With concat=False and heads=1 in the last layer, the output dim is hidden_dim
#         self.output_layer = nn.Linear(self.hparams.hidden_dim, self.hparams.output_dim)
#         self.sigmoid = nn.Sigmoid() # Sigmoid activation to bound output between 0 and 1

#         # --- Loss Function ---
#         self.loss_fn = nn.MSELoss() # Mean Squared Error for regression

#     def forward(self, data):
#         """
#         Defines the forward pass of the model.

#         Args:
#             data (torch_geometric.data.Data or torch_geometric.data.Batch):
#                   Input graph data.

#         Returns:
#             torch.Tensor: Node predictions (scores), shape [num_nodes, output_dim].
#         """
#         x, edge_index = data.x, data.edge_index

#         # 1. Apply initial embedding
#         x = self.input_embed(x)
#         x = F.relu(x)
#         x = F.dropout(x, p=self.hparams.dropout, training=self.training)

#         # 2. Apply GAT layers
#         for i, layer in enumerate(self.gat_layers):
#             x = layer(x, edge_index)
#             # Apply activation and dropout after GAT layers (except maybe the very last one before output)
#             if i < self.hparams.n_layers -1 : # Don't apply ReLU/dropout right before the final linear layer
#                  x = F.relu(x)
#                  x = F.dropout(x, p=self.hparams.dropout, training=self.training)

#         # 3. Apply final output layer and Sigmoid
#         predictions = self.output_layer(x)
#         predictions = self.sigmoid(predictions) # Apply sigmoid here to bound output to [0, 1]
#         return predictions

#     def _calculate_accuracy(self, y_hat, y, batch_vector):
#         """
#         Calculates the custom accuracy metric for a batch.

#         Args:
#             y_hat (torch.Tensor): Model predictions [num_nodes_in_batch].
#             y (torch.Tensor): True labels [num_nodes_in_batch].
#             batch_vector (torch.Tensor): Maps nodes to graphs [num_nodes_in_batch].

#         Returns:
#             torch.Tensor: Average accuracy across graphs in the batch (scalar).
#         """
#         y_true_rounded = torch.round(y)
#         y_pred_rounded = torch.round(y_hat)

#         accuracies = []
#         # Iterate through unique graph indices in the batch
#         for graph_idx in torch.unique(batch_vector):
#             mask = (batch_vector == graph_idx)
#             nodes_in_graph = mask.sum().item()

#             if nodes_in_graph > 0:
#                 correct_predictions = (y_true_rounded[mask] == y_pred_rounded[mask]).sum().item()
#                 graph_accuracy = correct_predictions / nodes_in_graph
#                 accuracies.append(graph_accuracy)
#             # else: # Handle case of graph with 0 nodes if possible? Should not happen with loader
#             #    accuracies.append(0.0) # Or skip

#         if not accuracies: # Handle empty batch or case where all graphs had 0 nodes
#             return torch.tensor(0.0, device=self.device)

#         # Calculate average accuracy across all graphs in the batch
#         # Convert list of Python floats to tensor
#         avg_accuracy = torch.tensor(accuracies, device=self.device).mean()
#         return avg_accuracy


#     def _shared_step(self, batch, batch_idx):
#         """Common logic for training, validation, and test steps."""
#         # Ensure labels `y` and batch vector exist
#         if not hasattr(batch, 'y') or not hasattr(batch, 'batch'):
#             raise ValueError("Batch object must have 'y' (labels) and 'batch' attributes.")

#         y_hat = self.forward(batch) # Get predictions: [N, 1]

#         # Ensure shapes match for loss calculation
#         # Remove last dimension if output_dim is 1
#         if self.hparams.output_dim == 1:
#             y_hat = y_hat.squeeze(-1) # -> [N]
#         y = batch.y.squeeze(-1)     # -> [N], assuming y was [N, 1]

#         # Ensure y_hat and y have the same shape
#         if y_hat.shape != y.shape:
#              raise RuntimeError(f"Shape mismatch: y_hat {y_hat.shape}, y {y.shape}")

#         loss = self.loss_fn(y_hat, y)
#         accuracy = self._calculate_accuracy(y_hat, y, batch.batch)

#         return loss, accuracy

#     def training_step(self, batch, batch_idx):
#         loss, accuracy = self._shared_step(batch, batch_idx)
#         # Log metrics using self.log (will automatically use WandbLogger)
#         self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=False, logger=True)
#         self.log('train_avg_acc', accuracy, on_step=True, on_epoch=True, prog_bar=False, logger=True)
#         return loss # Return loss for automatic optimization

#     def validation_step(self, batch, batch_idx):
#         loss, accuracy = self._shared_step(batch, batch_idx)
#         self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
#         self.log('val_avg_acc', accuracy, on_step=False, on_epoch=True, prog_bar=True, logger=True)
#         return loss # Optional: return loss for aggregation if needed

#     def test_step(self, batch, batch_idx):
#         loss, accuracy = self._shared_step(batch, batch_idx)
#         self.log('test_loss', loss, on_step=False, on_epoch=True, logger=True)
#         self.log('test_avg_acc', accuracy, on_step=False, on_epoch=True, logger=True)
#         return loss # Optional

#     def configure_optimizers(self):
#         optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
#         return optimizer

# dataset_path = 'DeepLSD/notebooks/graph_dataset'
# os.makedirs(dataset_path, exist_ok=True)
# print(f"Initializing dataset. Processing data if needed...")
# dataset = LineGraphRegressionDataset(root=dataset_path, image_data=image_data)
# print(f"Dataset ready. Number of graphs: {len(dataset)}")
# first_graph = dataset[0]
# print(f"Number of nodes: {first_graph.num_nodes}")
# print(f"Node features shape: {first_graph.x.shape}")
# print(f"Node labels shape: {first_graph.y.shape}")
# print(f"Original image key: {first_graph.image_key}")


# data_module_root = "/Users/matinurdu/Desktop/ETH_Zürich/FS25/3D-Vision/DeepLSD/notebooks/graph_dataset" # Choose a directory for the module's use
# batch_s = 1 # Small batch size for small dataset ToDo: for some reason higher batch_sizes confuses module
# train_val_test_ratio = (0.6,0.2,0.2)
# load_precomputed = False # Set to True to load if 'processed_graphs.pkl' exists in data_module_root

# # 2. Instantiate the DataModule
# print(f"\n--- Initializing DataModule ---")
# # Ensure image_data is defined in your scope before this line
# if 'image_data' not in locals() or not isinstance(image_data, dict):
#      raise NameError("The 'image_data' dictionary is not defined. Load your data first.")

# data_module = LineGraphDataModule(
#     image_data=image_data,
#     root=data_module_root,
#     load_precomputed_dataset=load_precomputed,
#     batch_size=batch_s,
#     train_val_test_split=train_val_test_ratio
    
# )

# # 3. Prepare Data (process/load underlying dataset)
# print(f"\n--- Preparing Data ---")
# data_module.prepare_data()

# # 4. Setup (load underlying dataset into module, create splits)
# print(f"\n--- Setting up DataLoaders ---")
# data_module.setup()
# data_module.save_dataset()

# # 5. Verification (similar to previous check)
# print(f"\n--- Verifying Train DataLoader ---")
# train_loader = data_module.train_dataloader()

# if train_loader is not None and len(train_loader) > 0:
#     print(f"Number of training batches: {len(train_loader)}")
#     # Get the first batch
#     first_batch = next(iter(train_loader))

#     print(f"Batch object type: {type(first_batch)}")
#     print(f"Number of graphs in first batch: {first_batch.num_graphs}")

#     # Extract the first graph from the batch
#     # The .get_example(0) method isolates the first graph's data from the batch
#     first_graph_in_batch = first_batch.get_example(0)

#     print("\n--- First Graph in First Training Batch ---")
#     print(f"Number of nodes: {first_graph_in_batch.num_nodes}") # Access num_nodes directly
#     print(f"Node features shape: {first_graph_in_batch.x.shape}")
#     print(f"Node labels shape: {first_graph_in_batch.y.shape}")
#     # Access custom attributes if they were added to the Data object (like image_key)
#     if hasattr(first_graph_in_batch, 'image_key'):
#         print(f"Original image key: {first_graph_in_batch.image_key}")
#     print(f"Edge index shape: {first_graph_in_batch.edge_index.shape}")

#     # You can also check validation/test loaders
#     # val_loader = data_module.val_dataloader()
#     # test_loader = data_module.test_dataloader()
#     # print(f"Validation batches: {len(val_loader) if val_loader else 0}")
#     # print(f"Test batches: {len(test_loader) if test_loader else 0}")

# elif len(data_module.train_dataset) == 0:
#      print("Training dataset is empty (likely due to small dataset size or split configuration).")
# else:
#     print("Train DataLoader could not be created or is empty.")
# wandb.login(key="89dd0dde666ab90e0366c4fec54fe1a4f785f3ef")
# wandb_logger = pl.loggers.WandbLogger(project="graph_structural_textural", log_model="all")

# model = GATRegressor(
#         input_dim=36864, # input feature dim is
#         hidden_dim=128,   # Smaller hidden dim might be better initially
#         n_heads=4,
#         n_layers=2,
#         dropout=0.2,
#         learning_rate=0.001,
#         batch_size=batch_s,
#     )
# checkpoint_callback = ModelCheckpoint(
#     dirpath="./checkpoints",
#     filename="gat-line-{epoch}-{val_loss:.2f}", # Include val_loss in name
#     save_top_k=1,        # Save top 2 models based on monitored metric
#     monitor="val_loss",  # Monitor validation loss
#     mode="min",          # Save models with minimum validation loss
#     save_last=True       # Also save the last epoch's checkpoint
# )
# trainer = pl.Trainer(
#         logger=wandb_logger, # Use the logger instance (can be None)
#         max_epochs=20,
#         accelerator="cpu", # Automatically choose accelerator (mps, cpu, gpu)
#         gradient_clip_val=1.0, # Added gradient clipping
#         log_every_n_steps=10, # Log slightly less often
#         callbacks=[checkpoint_callback],
#         # precision="16-mixed" # Optional: Use mixed precision if running on GPU for speed/memory
#         # detect_anomaly=True # Optional: Enable anomaly detection during training for debugging NaNs
#     )

# # 6. Train the model
# print("Starting training...")
# trainer.fit(model, datamodule=data_module)

# # 7. (Optional) Test the model
# print("Starting testing...")
# trainer.test(model, datamodule=data_module)

# # 8. Finish wandb run
# wandb.finish()
# print("Training and logging finished.")

# # --- Example: How to Initialize and Train ---

# # if __name__ == '__main__':
# #     # 1. Make sure wandb is logged in (run `wandb login` in your terminal)
# #
# #     # 2. Instantiate your DataModule (assuming it's defined and ready)
# #     # Example placeholder - replace with your actual data loading
# #     # data_module = LineGraphDataModule(...)
# #     # data_module.prepare_data()
# #     # data_module.setup()
# #     # --- Placeholder DataModule Setup ---
# #     class PlaceholderDataModule(pl.LightningDataModule):
# #         def __init__(self, batch_size=2):
# #             super().__init__()
# #             self.batch_size = batch_size
# #             # Create dummy data for demonstration
# #             self.data_list = []
# #             num_graphs = 9
# #             nodes_per_graph = [5, 8, 3, 6, 4, 7, 5, 9, 6]
# #             input_features = 36864
# #             for i in range(num_graphs):
# #                 n = nodes_per_graph[i]
# #                 x = torch.randn(n, input_features)
# #                 # Ensure fully connected edge_index
# #                 if n > 1:
# #                     adj = torch.ones((n, n), dtype=torch.long)
# #                     adj.fill_diagonal_(0)
# #                     edge_index, _ = torch_geometric.utils.dense_to_sparse(adj)
# #                 else:
# #                     edge_index = torch.empty((2, 0), dtype=torch.long)
# #                 y = torch.rand(n, 1) # Scores between 0 and 1
# #                 self.data_list.append(torch_geometric.data.Data(x=x, edge_index=edge_index, y=y, num_nodes=n))
# #             self.train_idx, self.val_idx, self.test_idx = [0,1], [2,3], [4,5,6,7,8] # Manual split for 2 train, 2 val
# #             self.train_dataset = [self.data_list[i] for i in self.train_idx]
# #             self.val_dataset = [self.data_list[i] for i in self.val_idx]
# #             self.test_dataset = [self.data_list[i] for i in self.test_idx]
# #
# #         def train_dataloader(self): return torch_geometric.loader.DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)
# #         def val_dataloader(self): return torch_geometric.loader.DataLoader(self.val_dataset, batch_size=self.batch_size)
# #         def test_dataloader(self): return torch_geometric.loader.DataLoader(self.test_dataset, batch_size=self.batch_size)
# #
# #     import torch_geometric # Needed for placeholder
# #     data_module = PlaceholderDataModule(batch_size=2) # Use batch_size >= 1
# #     # --- End Placeholder ---
# #
# #     # 3. Initialize Wandb Logger
# #     wandb_logger = pl.loggers.WandbLogger(project="GAT-Line-Regression-Simple", log_model="all") # Log model checkpoints
# #
# #     # 4. Initialize the Model
# #     # IMPORTANT: Set input_dim correctly based on your flattened features
# #     model = GATRegressor(
# #         input_dim=36864, # Or whatever your actual feature dim is
# #         hidden_dim=64,   # Smaller hidden dim might be better initially
# #         n_heads=4,
# #         n_layers=2,
# #         dropout=0.2,
# #         learning_rate=0.001
# #     )
# #
# #     # 5. Initialize PyTorch Lightning Trainer
# #     # Adjust trainer settings as needed (GPU, epochs, etc.)
# #     trainer = pl.Trainer(
# #         logger=wandb_logger,
# #         max_epochs=20, # Adjust number of epochs
# #         accelerator="auto", # Use GPU if available ("gpu") or CPU ("cpu")
# #         devices="auto",     # Use all available devices of the selected accelerator
# #         # gradient_clip_val=0.5, # Optional gradient clipping
# #         log_every_n_steps=10 # Log less frequently if steps are fast
# #     )
# #
# #     # 6. Train the model
# #     print("Starting training...")
# #     trainer.fit(model, datamodule=data_module)
# #
# #     # 7. (Optional) Test the model
# #     print("Starting testing...")
# #     trainer.test(model, datamodule=data_module)
# #
# #     # 8. Finish wandb run
# #     wandb.finish()
# #     print("Training and logging finished.") """